# Interaction Radius Sweep

### Motivation
balance point (optimal rejection_cost peaks in the middle rather than always at maximum)
requires cautious agents to be *specifically* penalised more than bold agents when conditions
get tight. Gender imbalance didn't do this — it stranded everyone equally.

A smaller `interaction_radius` should create the differential penalty we need:
- **Bold agents** match on first or second contact — small radius is fine.
- **Cautious agents** need to accumulate observations across many candidates before committing.
  With a small radius, they see fewer people per step and may never build enough evidence
  to commit before good candidates are taken.

### What we test
We sweep `interaction_radius` in both the **renewable** and **permanent** markets,
holding everything else at baseline. For each market type we look at:
1. How quality and matched_fraction change with radius — per rejection_cost level
2. Whether the balance point (peak in total welfare at intermediate rejection_cost)
   appears at small radii
3. Whether the dominant strategy shifts as radius shrinks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dating_market import DatingMarket, Agent

## 1. Permanent-market patch (same as PermanentMatchExperiment)

In [ ]:
def _rank_of(agent, partner_id):
    if partner_id not in agent._cnt:
        return None
    means = {}
    for oid, cnt in agent._cnt.items():
        n = min(cnt, agent.memory_depth)
        if n > 0:
            means[oid] = float(np.mean(agent._buf[oid][:n]))
    ranked = sorted(means, key=lambda x: -means[x])
    try:
        return ranked.index(partner_id)
    except ValueError:
        return None


def _permanent_act(self):
    self.partner = None
    target = self._choose_target()
    if target is None:
        return
    other = self.model.subjects[target]
    if other.consider_proposal(self.id):
        self.partner = target
        other.partner = self.id
        self.engaged_until = np.inf
        other.engaged_until = np.inf
        self.utility += 1.0
        self.match_rank  = _rank_of(self, target)
        other.match_rank = _rank_of(other, self.id)
        self.match_time  = self.model.t
        other.match_time = self.model.t
        self.model.grid[self.pos]  = DatingMarket.EMPTY
        self.model.grid[other.pos] = DatingMarket.EMPTY
    else:
        self.utility -= self.rejection_cost


def _init_tracking(market):
    for a in market.subjects:
        a.match_rank = None
        a.match_time = None


# will be toggled on/off per experiment cell
_original_act = Agent.act

## 2. Run functions

In [ ]:
def run_renewable(
    rejection_cost, rationality, interaction_radius,
    seed, n=240, n_grid=50, n_steps=200, burn_in=80,
    gender_balance=0.5, memory_depth=8, relation_threshold=0.6,
):
    """Single run of the renewable (finite relationship_length) market."""
    Agent.act = _original_act   # make sure patch is off
    m = DatingMarket(
        n_grid=n_grid, interaction_std=0.5,
        interaction_radius=interaction_radius,
        relationship_length=10, seed=seed,
    )
    m.add_agents(
        n, gender_balance=gender_balance,
        rejection_cost=rejection_cost, rationality=rationality,
        relation_threshold=relation_threshold,
        memory_depth=memory_depth, move_prob=0.5,
    )
    q, mf = [], []
    for t in range(n_steps):
        m.step()
        if t >= burn_in:
            vals = [m.compatibility(a.id, a.partner)
                    for a in m.subjects if not a.is_single]
            q.append(float(np.mean(vals)) if vals else np.nan)
            mf.append(float(np.mean([not a.is_single for a in m.subjects])))
    return float(np.nanmean(q)), float(np.nanmean(mf))


def run_permanent_radius(
    rejection_cost, rationality, interaction_radius,
    seed, n=240, n_grid=50, max_steps=400, stall_window=20,
    gender_balance=0.5, memory_depth=8, relation_threshold=0.6,
):
    """Single run of the permanent market with a given interaction_radius."""
    Agent.act = _permanent_act   # patch on
    m = DatingMarket(
        n_grid=n_grid, interaction_std=0.5,
        interaction_radius=interaction_radius,
        relationship_length=10, seed=seed,
    )
    m.add_agents(
        n, gender_balance=gender_balance,
        rejection_cost=rejection_cost, rationality=rationality,
        relation_threshold=relation_threshold,
        memory_depth=memory_depth, move_prob=0.5,
    )
    _init_tracking(m)

    prev_matched, stall_count = 0, 0
    for _ in range(max_steps):
        m.step()
        now = sum(not a.is_single for a in m.subjects)
        stall_count = stall_count + 1 if now == prev_matched else 0
        prev_matched = now
        if stall_count >= stall_window:
            break

    matched = [a for a in m.subjects if not a.is_single]
    quality = float(np.mean([m.compatibility(a.id, a.partner)
                              for a in matched])) if matched else np.nan
    mf = len(matched) / len(m.subjects)
    return quality, mf


def sweep_radius(
    run_fn,
    radii, rejection_costs,
    rationality=6.0, seeds=range(8), **kw
):
    """
    Sweep interaction_radius × rejection_cost and average over seeds.
    Returns a DataFrame with columns:
      interaction_radius, rejection_cost, mean_quality, mean_matched_fraction
    """
    rows = []
    for r in radii:
        for rc in rejection_costs:
            results = [run_fn(rejection_cost=rc, rationality=rationality,
                              interaction_radius=r, seed=s, **kw)
                       for s in seeds]
            qs, mfs = zip(*results)
            rows.append({
                "interaction_radius":  r,
                "rejection_cost":      rc,
                "mean_quality":        float(np.nanmean(qs)),
                "matched_fraction":    float(np.nanmean(mfs)),
            })
    return pd.DataFrame(rows)

## 3. Run the sweeps

We test radii from 1 (near-neighbour only) to 8 (wide neighbourhood).
The baseline in both markets was 5.

In [ ]:
RADII          = [1, 2, 3, 5, 8]
REJECTION_COSTS = [0.2, 0.5, 1.0, 2.0, 4.0]
SEEDS          = range(8)

print("Running renewable sweep...")
df_renewable = sweep_radius(
    run_renewable, RADII, REJECTION_COSTS, seeds=SEEDS
)

print("Running permanent sweep...")
df_permanent = sweep_radius(
    run_permanent_radius, RADII, REJECTION_COSTS, seeds=SEEDS
)

print("Done.")

## 4. Plot quality and matched fraction vs rejection_cost — one panel per radius

This shows whether the quality advantage of cautious agents *shrinks* as radius decreases,
and whether their matched_fraction penalty *grows*.

In [ ]:
def plot_radius_sweep(df, title):
    fig, axes = plt.subplots(2, len(RADII), figsize=(16, 7), sharey="row")
    for col, r in enumerate(RADII):
        sub = df[df["interaction_radius"] == r]
        axes[0, col].plot(sub["rejection_cost"], sub["mean_quality"], marker="o")
        axes[0, col].set_title(f"radius={r}")
        axes[0, col].grid(alpha=0.3)
        if col == 0:
            axes[0, col].set_ylabel("mean quality (↑ better)")

        axes[1, col].plot(sub["rejection_cost"], sub["matched_fraction"],
                          marker="s", color="C1")
        axes[1, col].set_xlabel("rejection_cost")
        axes[1, col].grid(alpha=0.3)
        if col == 0:
            axes[1, col].set_ylabel("matched fraction")

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

plot_radius_sweep(df_renewable, "Renewable market — quality & matched fraction vs rejection_cost")
plot_radius_sweep(df_permanent, "Permanent market — quality & matched fraction vs rejection_cost")

## 5. welfare metric — does the balance point appear at small radii?

For each (radius, single_utility) combination, plot total welfare vs rejection_cost.
Look for curves that peak in the middle rather than at the right edge.

In [ ]:
def plot_welfare_by_radius(df, title, single_utilities=[0.0, 0.3, 0.6, 0.9]):
    fig, axes = plt.subplots(
        1, len(RADII), figsize=(16, 4.5), sharey=True
    )
    for ax, r in zip(axes, RADII):
        sub = df[df["interaction_radius"] == r].copy()
        for su in single_utilities:
            sub["welfare"] = (
                sub["matched_fraction"] * sub["mean_quality"]
                + (1 - sub["matched_fraction"]) * su
            )
            ax.plot(sub["rejection_cost"], sub["welfare"],
                    marker="o", label=f"u_s={su}")
        ax.set_title(f"radius={r}")
        ax.set_xlabel("rejection_cost  (← bold | cautious →)")
        ax.grid(alpha=0.3)
        if r == RADII[0]:
            ax.set_ylabel("total welfare")
        if r == RADII[-1]:
            ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

plot_welfare_by_radius(
    df_renewable,
    "Renewable market — total welfare (does balance point appear at small radius?)"
)
plot_welfare_by_radius(
    df_permanent,
    "Permanent market — total welfare (does balance point appear at small radius?)"
)

## 6. Optimal rejection_cost table — does it shift left as radius shrinks?

If the balance point appears, optimal rejection_cost should move away from 4.0
toward lower values as radius decreases and single_utility increases.

In [ ]:
def print_optimal_rc(df, label, single_utilities=[0.0, 0.3, 0.6, 0.9]):
    print(f"\nOptimal rejection_cost — {label}")
    header = f"{'':12s}" + "".join(f"  u_s={su}" for su in single_utilities)
    print(header)
    for r in RADII:
        sub = df[df["interaction_radius"] == r].copy()
        row = f"radius={r:<5}"
        for su in single_utilities:
            sub["welfare"] = (
                sub["matched_fraction"] * sub["mean_quality"]
                + (1 - sub["matched_fraction"]) * su
            )
            best = sub.loc[sub["welfare"].idxmax(), "rejection_cost"]
            row += f"  {best:>7.1f}"
        print(row)

print_optimal_rc(df_renewable, "renewable market")
print_optimal_rc(df_permanent, "permanent market")

## 7. Strategy competition at small radius

Does the ranking of the four strategies (bold/cautious × rational/random) change
when the radius is very small? Run the mixed market at radius=1 and radius=5
and compare.

In [ ]:
strat_grid = [
    dict(rejection_cost=0.2, rationality=50.0, label="bold-rational"),
    dict(rejection_cost=4.0, rationality=50.0, label="cautious-rational"),
    dict(rejection_cost=0.2, rationality=1.0,  label="bold-random"),
    dict(rejection_cost=4.0, rationality=1.0,  label="cautious-random"),
]


def run_mixed_renewable(strategies, seed, radius, n_per=80, n_grid=50,
                        n_steps=200, burn_in=80, gb=0.5, thr=0.6):
    Agent.act = _original_act
    m = DatingMarket(n_grid=n_grid, interaction_std=0.5,
                     interaction_radius=radius,
                     relationship_length=10, seed=seed)
    sid_label = {}
    for st in strategies:
        sid = m.add_agents(n_per, gender_balance=gb,
                           rejection_cost=st["rejection_cost"],
                           rationality=st["rationality"],
                           relation_threshold=thr, memory_depth=8,
                           move_prob=0.5, label=st["label"])
        sid_label[sid] = st["label"]
    acc = {sid: [] for sid in sid_label}
    for t in range(n_steps):
        m.step()
        if t >= burn_in:
            for sid in sid_label:
                mem = [a for a in m.subjects
                       if a.strategy_id == sid and not a.is_single]
                if mem:
                    acc[sid].append(np.mean(
                        [m.compatibility(a.id, a.partner) for a in mem]))
    return {sid_label[sid]: (np.nanmean(v) if v else np.nan)
            for sid, v in acc.items()}


def run_mixed_permanent_radius(strategies, seed, radius, n_per=80, n_grid=50,
                               max_steps=400, stall_window=20, gb=0.5, thr=0.6):
    Agent.act = _permanent_act
    m = DatingMarket(n_grid=n_grid, interaction_std=0.5,
                     interaction_radius=radius,
                     relationship_length=10, seed=seed)
    sid_label = {}
    for st in strategies:
        sid = m.add_agents(n_per, gender_balance=gb,
                           rejection_cost=st["rejection_cost"],
                           rationality=st["rationality"],
                           relation_threshold=thr, memory_depth=8,
                           move_prob=0.5, label=st["label"])
        sid_label[sid] = st["label"]
    _init_tracking(m)
    prev_matched, stall_count = 0, 0
    for _ in range(max_steps):
        m.step()
        now = sum(not a.is_single for a in m.subjects)
        stall_count = stall_count + 1 if now == prev_matched else 0
        prev_matched = now
        if stall_count >= stall_window:
            break
    out = {}
    for sid, label in sid_label.items():
        members = [a for a in m.subjects
                   if a.strategy_id == sid and not a.is_single]
        n_total = sum(1 for a in m.subjects if a.strategy_id == sid)
        out[label] = {
            "mean_quality":     float(np.mean([m.compatibility(a.id, a.partner)
                                               for a in members])) if members else np.nan,
            "matched_fraction": len(members) / n_total if n_total else np.nan,
        }
    return out


compare_radii = [1, 2, 5]
seeds = range(8)

print("Mixed-strategy quality by radius — RENEWABLE market")
print(f"{'strategy':20s}" + "".join(f"  radius={r}" for r in compare_radii))
labels = [st["label"] for st in strat_grid]
ren_by_radius = {}
for r in compare_radii:
    results = pd.DataFrame([run_mixed_renewable(strat_grid, s, radius=r)
                            for s in seeds]).mean()
    ren_by_radius[r] = results
for lbl in labels:
    row = f"{lbl:20s}"
    for r in compare_radii:
        row += f"  {ren_by_radius[r][lbl]:>9.3f}"
    print(row)

print("\nMixed-strategy quality by radius — PERMANENT market")
print(f"{'strategy':20s}" + "".join(f"  radius={r}" for r in compare_radii))
perm_by_radius = {}
for r in compare_radii:
    rows = []
    for s in seeds:
        res = run_mixed_permanent_radius(strat_grid, s, radius=r)
        rows.append({lbl: res[lbl]["mean_quality"] for lbl in res})
    perm_by_radius[r] = pd.DataFrame(rows).mean()
for lbl in labels:
    row = f"{lbl:20s}"
    for r in compare_radii:
        row += f"  {perm_by_radius[r][lbl]:>9.3f}"
    print(row)

## 8. Summary visualisation — strategy quality ranking across radii

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, by_radius, title in zip(
    axes,
    [ren_by_radius, perm_by_radius],
    ["Renewable market", "Permanent market"],
):
    for lbl in labels:
        ax.plot(compare_radii,
                [by_radius[r][lbl] for r in compare_radii],
                marker="o", label=lbl)
    ax.set_title(title)
    ax.set_xlabel("interaction_radius  (← tight | loose →)")
    ax.set_ylabel("mean match quality")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(compare_radii)

fig.suptitle("Does the dominant strategy shift as interaction radius shrinks?", fontsize=12)
plt.tight_layout()
plt.show()